# sae-arabic: MARBERT SAE training run (Colab)

Runs the Phase 1-2 pipeline on a Colab GPU: Arabic dialect data -> MARBERT layer activations -> SAE training -> checkpoints + report.

## Before running
- Runtime -> **Change runtime type** -> Accelerator: **T4 GPU**
- Have ready:
  - Your **HuggingFace** token (`hf_...`) from huggingface.co -> Settings -> Access Tokens
  - (The repo is public, so no GitHub token is needed.)

In [ ]:
!nvidia-smi | head -3
import os
import getpass

def _clean(prompt):
    return "".join((getpass.getpass(prompt=prompt) or "").split())

os.environ["HF_TOKEN"] = _clean("HuggingFace token (hf_...): ")
print("HF token captured")

In [ ]:
import io
import os
import shutil
import zipfile

import requests

r = requests.get(
    "https://api.github.com/repos/Yousef13133/sae-arabic/zipball/main",
    timeout=120,
)
assert r.status_code == 200, f"repo download failed: HTTP {r.status_code}"
shutil.rmtree("/content/sae-arabic", ignore_errors=True)
zipfile.ZipFile(io.BytesIO(r.content)).extractall("/content/sae-arabic")
repo_dir = os.path.join("/content/sae-arabic", os.listdir("/content/sae-arabic")[0])
os.chdir(repo_dir)
!pip install -e .

If cell 3 failed to download the repo, check your Colab internet connection. The cell below runs the pipeline.

In [ ]:
import os
os.chdir(os.path.join("/content/sae-arabic", os.listdir("/content/sae-arabic")[0]))

# 200 sentences, layer 6, 10k steps. ~10-25 min on a T4.
!python scripts/train_real.py --num-samples 200 --layer 6 --num-steps 10000 --out-dir /content/real_run

print("\n=== report.json ===")
!cat /content/real_run/report.json

print("\n=== evaluation.json ===")
!python scripts/evaluate.py --checkpoint /content/real_run/checkpoints/final.pt --activations-dir /content/real_run/activations --out /content/real_run/evaluation.json
!cat /content/real_run/evaluation.json

In [ ]:
!zip -r /content/real_run.zip /content/real_run > /dev/null && echo "created /content/real_run.zip"
print("Download via left file panel: real_run/ or real_run.zip")